# Kapitel 32.9 - FastAPI Global Exception Middleware

In diesem Notebook behandeln wir die zentrale Exception-Handling-Strategie in FastAPI Schritt fuer Schritt.

# Lernziele

- die FastAPI-Exception-Handler-Kette verstehen
- Domain-Fehlertypen auf HTTP-Antworten abbilden
- ein konsistentes Fehler-Response-Format erstellen

# Voraussetzungen

- Kapitel 21 und 32.6
- Grundkenntnisse in FastAPI

# Theorie

Globales Exception Handling reduziert verstreute try/except-Bloecke auf Endpoint-Ebene.
Dieser Ansatz verbessert Wartbarkeit und Observability.

# Erklaerung

Fehlerhierarchie: Validation -> NotFound -> Auth -> Generic.
Jede Fehlerklasse wird auf ein einheitliches Response-Schema abgebildet.

# Syntax

```python
@app.exception_handler(MyError)
async def handler(request, exc):
    return JSONResponse(...)
```

# Merke

- Ein einheitliches Fehlerformat vereinfacht den Client-Code.
- Unerwartete Fehler werden sauber als 500 isoliert.

# Parameter

- request context
- exception instance
- status code map

# Rueckgabewert

JSON-Fehlerantwort, z. B.: {ok:false,error:'...'}

In [ ]:
# Beispiel 1: Domain-Fehlerklassen
class AppError(Exception):
    pass

class ValidationError(AppError):
    pass

In [ ]:
# Beispiel 2: HTTP-Mapping-Funktion
def map_error(exc):
    if isinstance(exc, ValidationError):
        return {'status': 400, 'error': str(exc)}
    return {'status': 500, 'error': 'internal server error'}

print(map_error(ValidationError('bad input')))

In [ ]:
# Beispiel 3: Konsistentes Antwortformat
def error_payload(message):
    return {'ok': False, 'error': message}

print(error_payload('course not found'))

# Praxisbeispiel

Referenzprojekt: 32_Architektur_und_Security_Patterns/projects/fastapi_clean_cqrs_jwt_sample/app/main.py

# Haeufige Fehler

1. In jedem Endpoint eigene try/except-Bloecke nutzen
2. Interne Exception-Meldungen direkt an den Client durchreichen
3. Auth-Fehler faelschlich als 500 zurueckgeben

# Best Practice

- Zentraler Handler
- Domain-specific exception type
- Structured logging

# Tipp

Den Error-Schema-Vertrag zwischen Frontend und Backend stabil halten.

# Uebung

Schreibe eine Beispiel-Mapping-Funktion fuer ValidationError und NotFoundError.

In [ ]:
# Loesung
def map_error_v2(exc):
    name = exc.__class__.__name__
    if name == 'ValidationError':
        return {'status': 400, 'ok': False, 'error': str(exc)}
    if name == 'NotFoundError':
        return {'status': 404, 'ok': False, 'error': str(exc)}
    return {'status': 500, 'ok': False, 'error': 'internal server error'}

print(map_error_v2(Exception('x')))

# Zusammenfassung

Globales Exception Handling schafft in FastAPI-Projekten eine nachhaltige Fehlerarchitektur.

# Weiterfuehrende Links

- FastAPI exception handlers
- RFC7807 problem details

## Technischer Tiefgang

Auf fortgeschrittenem Niveau steht nicht nur Funktionalitaet, sondern die technische Nachvollziehbarkeit im Vordergrund.
Begriffe, Risiken, Betriebsaspekte und Qualitaetskriterien werden explizit gemacht, damit Entscheidungen reproduzierbar bleiben.

## Zentrale Fachbegriffe

Design Constraint
Trade-off
Failure Mode
Observability Signal
Quality Gate
Regression Risk

In [ ]:
# Zusatzbeispiel: Priorisierung technischer Risiken
risks=[{"name":"regression","score":9},{"name":"operability","score":8},{"name":"complexity","score":7}]
for r in sorted(risks,key=lambda x:x["score"],reverse=True):
    print(r["name"], r["score"])

## Fallstudie (Praxis)

Praxis-Szenario: Ein Team liefert ein Feature aus, das lokal stabil wirkt, in Staging jedoch sporadisch ausfaellt.
Beschreibe ein strukturiertes Vorgehen mit Hypothesen, Messsignalen, Gegenbeweisen und finaler Ursachenbehebung.

## Haeufige Fehler und Debugging-Checkliste

- Sind Eingaben, Konfiguration und Randbedingungen explizit validiert?
- Sind reproduzierbare Schritte fuer den Fehler dokumentiert?
- Wurden Logs, Metriken und Tests gemeinsam ausgewertet?
- Ist die Korrektur durch einen neuen Test dauerhaft abgesichert?
- Gibt es eine kurze Lessons-Learned-Notiz fuer das Team?

## Pruefungsfragen und Kurzloesungen

1. Warum ist technische Reproduzierbarkeit fuer Qualitaet entscheidend?
Kurzloesung: Nur reproduzierbare Befunde lassen sich verifizieren, beheben und regressionssicher absichern.
2. Was ist ein typischer Fehler bei schnellen Feature-Releases?
Kurzloesung: Betriebs- und Risikoaspekte werden zu spaet betrachtet.
3. Welche Rolle hat ein Quality Gate?
Kurzloesung: Es verhindert unsichere Releases durch verbindliche Mindestkriterien.